# Simulate table updates
Appends new rows to a dedicated table, standing in for a pipeline load, so freshness checks have a recently updated table to evaluate. Run before each freshness check.

In [0]:
%sql
-- Create the table on the first run only, then append rows on every run
create table if not exists validation_sweep.dev.daily_orders as
select * from samples.tpch_sf1.orders where o_orderkey <= 1000;

insert into validation_sweep.dev.daily_orders
select * from samples.tpch_sf1.orders limit 100;

In [0]:
# Confirm the append was recorded as a data change, using the logic from explore_table_freshness
from pyspark.sql import functions as F

history = spark.sql("describe history validation_sweep.dev.daily_orders")

data_changes = history.where(
    F.col("operationMetrics").getItem("numOutputRows").cast("long") > 0
)

latest = data_changes.orderBy(F.col("version").desc()).first()
print(f"Last data change: version {latest['version']}, {latest['operation']}, "
      f"{latest['operationMetrics']['numOutputRows']} rows at {latest['timestamp']} UTC")